### Fine-Tuning a model to predict labels for LiDAR Point Clouds - 3D Semantic Segmentation - Waymo Open Dataset

#### By Jacob Igo

In [ ]:
import gcsfs
from google.cloud import storage
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import pandas as pd
import tensorflow as tf
import open3d as o3d
import numpy as np
import gc
import os
import random

import semseg_functions

import torch
import torch.nn as nn
import torch.nn.functional as F





In [10]:

#get google cloud token
os.environ["CLOUDSDK_CONFIG"] = "/home/jacob/.config/gcloud"

import subprocess
token = subprocess.check_output(
    ["/usr/bin/gcloud", "auth", "print-access-token"]
).decode().strip()

from datetime import datetime, timezone, timedelta
fs = pafs.GcsFileSystem(access_token=token, credential_token_expiration=datetime.now(timezone.utc) + timedelta(hours=1))

folders = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/"))
lidar_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar/"))
seg_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar_segmentation/"))
calib_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar_calibration/"))


In [11]:
def frame_loader(basefile, timestamp, sample_size=4096, seed=42):

    calib_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar_calibration/{basefile}", filesystem=fs)
    calib_df = calib_pq.read_row_group(0).to_pandas()

    # seg_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar_segmentation/{basefile}", filesystem=fs)
    # lidar_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar/{basefile}", filesystem=fs)

    lidar_df = semseg_functions.timestamp_aligner(f"waymo_open_dataset_v_2_0_0/training/lidar/{basefile}", timestamp=timestamp)
    seg_df = semseg_functions.timestamp_aligner(f"waymo_open_dataset_v_2_0_0/training/lidar_segmentation/{basefile}", timestamp=timestamp)

    g_coords_1, masked_labels_1 = semseg_functions.laser_process(laser_num=1, df=lidar_df, df_rgc=calib_df, df_seg=seg_df, timestamp=timestamp)

    #sampling, as model expects fixed input size
    rng = np.random.default_rng(seed=seed)
    random_indices = rng.integers(low=0, high=len(g_coords_1), size=sample_size)
    random_points = g_coords_1[random_indices]
    random_labels = masked_labels_1[random_indices]

    #normalizing, keeps points on a consistent scale for easier learning
    points_min = random_points.min(axis=0)
    points_max = random_points.max(axis=0)

    random_points_norm = (random_points - points_min) / (points_max - points_min)

    return random_points_norm, random_labels





In [ ]:

#loads a batch from of random files from either training or validation sets (after split)
def batch_loader(specified_set, seed=42):
    # random.seed(seed)
    # sample_list = random.sample(specified_set, k=K)

    point_clouds = []
    label_sets = []

    for sample in specified_set:

        file_name, timestamp = sample
        base_file_name = os.path.basename(file_name)
        points, labels = frame_loader(basefile=base_file_name, timestamp=timestamp)

        #scanning the labels and points to see if they are valid instances
        unique_labels, label_counts_by_index = np.unique(labels, return_counts=True)
        #print("Unique labels per point cloud: ", (len(unique_labels)))
        # print("How many times does each label occur: ", label_counts_by_index)

        if len(unique_labels) < 2:
            raise ValueError("not enough labels")
        
        point_clouds.append(points)
        label_sets.append(labels)

    
    # points_labels = np.vstack((point_clouds, label_sets))
    #stack point clouds and labels from each sample into two distinct lists
    stacked_points = np.stack(point_clouds)
    stacked_labels = np.stack(label_sets)

    
    return stacked_points, stacked_labels


    

#splitting the data up on an indexed folder   
def train_val_split(index, train_split=80, seed=42):

    training_file_list = []
    val_file_list = []

    random.seed(seed)
    set_of_files = set()
    
    #extract unique file-basenames
    for file, _ in index:
        base_file = os.path.basename(file)
        set_of_files.add(base_file)

    #shuffle them for random order
    list_of_files = list(set_of_files)
    shuffled_files = random.sample(list_of_files, len(set_of_files))

    #split them up (train and validation folders)
    counter=0
    train_split_percentage = train_split / 100
    for unique_file in shuffled_files:
        if counter < len(shuffled_files) * train_split_percentage:
            training_file_list.append(unique_file)
            counter += 1
        else:
            val_file_list.append(unique_file)

    # return training_file_list, val_file_list 
    # adding the frames to the files after splitting for the dataloader
    training_frames = []
    val_frames = []

    for split_file, timestamp in index:
        base_split_file = os.path.basename(split_file)
        if base_split_file in training_file_list:
            training_frames.append((base_split_file, timestamp))
        elif base_split_file in val_file_list:
            val_frames.append((base_split_file, timestamp))
        else:
            raise ValueError("This basefile in the main index was NOT in the training or test file lists")

    return training_frames, val_frames


    


#testing on seg
seg_training = semseg_functions.folder_file_indexer(folder="training/lidar_segmentation/", start_folder_index=0, end_folder_index=2)
    
training, validating = train_val_split(seg_training)
points, labels = batch_loader(K=5, specified_set=training)

print(f"Shape of stacked points: {points.shape}")
print(f"Shape of stacked labels: {labels.shape}")



Unique labels per point cloud:  17
Unique labels per point cloud:  13
Unique labels per point cloud:  13
Unique labels per point cloud:  16
Unique labels per point cloud:  13
Shape of stacked points: (5, 4096, 3)
Shape of stacked labels: (5, 4096)


In [13]:
del seg_training
del training, validating, points, labels

In [14]:
#creating the PointNetSeg architecture

"""Describe each point on its own (64 matrix), squeeze all points into a single scene-summary (1024 matrix for 4096 points), 
hand that summary back to every point (concat the 64 and 1024 matrices), then let each point pick its own label knowing both 
itself and the whole scene (downsize convolutions (with BatchNorm + ReLU)) — and because both the "describe" and "summarize" 
steps ignore order, the whole thing works on an unordered bag of points.
"""

class PointNetSeg(nn.Module):
    def __init__(self, num_classes=23):
        super(PointNetSeg, self).__init__()

        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        #local 64 features per point + global 1024 (max) features per scene
        self.conv4 = nn.Conv1d(1024+64, 512, 1)
        self.conv5 = nn.Conv1d(512, 256, 1)
        self.conv6 = nn.Conv1d(256, 128, 1)
        self.conv7 = nn.Conv1d(128, num_classes, 1)

        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)
        self.bn6 = nn.BatchNorm1d(128)

    def forward(self, x):

        N = x.size(2)

        x = F.relu(self.bn1(self.conv1(x)))
        local_feature = x
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        #max pool for global features over point axis for the scene
        global_feat = torch.max(x, 2, keepdim=True)[0]
        #tiling global features onto every point, then concatenate with local features per point
        global_feat = global_feat.repeat(1, 1, N)         # (B, 1024, N)
        x = torch.cat([local_feature, global_feat], dim=1)

        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.relu(self.bn6(self.conv6(x)))
        x = self.conv7(x)

        return x

In [ ]:
#testing the model - batch overfit

seg_training_batch = semseg_functions.folder_file_indexer(folder="training/lidar_segmentation/", start_folder_index=0, end_folder_index=2)
    
training_batch, validating_batch = train_val_split(seg_training_batch)
points_1, labels_1 = batch_loader(K=1, specified_set=training_batch)

#setting point orientation for model training
points_tensor = torch.tensor(points_1, dtype=torch.float32)
labels_tensor = torch.tensor(labels_1, dtype=torch.int64)
points_tensor = points_tensor.transpose(1, 2)

STEPS = 200

model = PointNetSeg()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
model.train()

for i in range(STEPS):

    optimizer.zero_grad()

    pred = model(points_tensor)     #pred shape: (1, 23, 4096) (after taking away transpose(1, 2).contigous)

    loss = criterion(pred, labels_tensor)

    loss.backward()

    optimizer.step()

    if i % 20 == 0:
        print(f"Step: {i}; === Loss: {loss.item():.4f}")



del seg_training_batch
del training_batch, validating_batch
del model

Unique labels per point cloud:  17
Step: 0; === Loss: 3.2028
Step: 20; === Loss: 0.4421
Step: 40; === Loss: 0.2599
Step: 60; === Loss: 0.1878
Step: 80; === Loss: 0.1302
Step: 100; === Loss: 0.1768
Step: 120; === Loss: 0.1135
Step: 140; === Loss: 0.0873
Step: 160; === Loss: 0.0783
Step: 180; === Loss: 0.2468


In [ ]:
def calculate_matrix(preds, targets, num_classes, ignore_index=None):
    """
    Calculates the mean Intersection over Union (mIoU).
    Args:
        preds (Tensor): Predicted mask of shape (B, H, W) or flattened.
        targets (Tensor): Ground truth mask of same shape.
        num_classes (int): Total number of semantic classes.
        ignore_index (int, optional): Class index to ignore (e.g., background/void).
    """
    # Flatten tensors to 1D
    preds = preds.contiguous().view(-1)
    targets = targets.contiguous().view(-1)
    
    # Filter out the ignore index if specified
    if ignore_index is not None:
        mask = targets != ignore_index
        preds = preds[mask]
        targets = targets[mask]

    # Calculate confusion matrix: rows = targets, cols = preds
    # bincount constructs a flat matrix of size num_classes^2
    indices = num_classes * targets + preds
    conf_matrix = torch.bincount(indices, minlength=num_classes**2)
    conf_matrix = conf_matrix.reshape(num_classes, num_classes)

    return conf_matrix

def calculate_iou_miou(conf_matrix, ignore_index=None):

    # Intersection is the diagonal elements (TP)
    intersection = torch.diag(conf_matrix)
    
    # Union = TP + FP + FN
    # Ground truth per class (axis 1) + Predictions per class (axis 0) - Intersection
    union = conf_matrix.sum(dim=1) + conf_matrix.sum(dim=0) - intersection

    # Calculate IoU per class
    # Use a small epsilon to avoid division by zero if a class is entirely absent
    iou = intersection.float() / (union.float() + 1e-10)
    
    # Exclude classes that are not present in either targets or predictions
    present_classes = (union > 0)
    if ignore_index != None:
        present_classes[ignore_index] = False
    if present_classes.sum() == 0:
        return None, torch.tensor(0.0), None
        
    mean_iou = iou[present_classes].mean()
    return iou, mean_iou, conf_matrix

In [ ]:
#count classes per points for Weighted Cross-Entropy Loss

def class_per_point_weights(frames, num_classes, K=20, ignore_index=None):

    running_class = torch.zeros(23)

    for frame_step in range(0, len(frames), K):

        _, labels = batch_loader(frames[frame_step:frame_step+K])
        labels_tensor = torch.tensor(labels, dtype=torch.int64)
        labels_flattened = labels_tensor.contiguous().view(-1)
        class_array = torch.bincount(labels_flattened, minlength=num_classes)

        running_class += class_array

    #ignore the 0 class
    if ignore_index is not None:
        mask = running_class > 0
        mask[ignore_index] = False
    

    #inverse median frequency
    total_counts = running_class.sum().float()
    frequencies = running_class.float() / total_counts
    median_freq = torch.median(frequencies[mask])

    class_weights = torch.zeros(23)
    class_weights[mask] = median_freq / frequencies[mask]

    #normalize
    class_weights[mask] = class_weights[mask] / class_weights[mask].mean()

    return class_weights


        

In [ ]:
#real training loop

seg_training_batch = semseg_functions.folder_file_indexer(folder="training/lidar_segmentation/", start_folder_index=0, end_folder_index=2)
training_batch, validating_batch = train_val_split(seg_training_batch)

train_shuffle_len = len(training_batch)
val_shuffle_len = len(validating_batch)

EPOCHS = 100
VAL_CHECK = 5
K=20

model = PointNetSeg()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

for epoch in range(EPOCHS):

    shuffled_training = random.sample(training_batch, train_shuffle_len)
    model.train()

    for step in range(0, train_shuffle_len, K):

        #drop the last chunk of K frames if the chunk is smaller than K
        if train_shuffle_len - step < K:
            break

        points, labels = batch_loader(specified_set=shuffled_training[step:step+K])
        points_tensor = torch.tensor(points, dtype=torch.float32)
        labels_tensor = torch.tensor(labels, dtype=torch.int64)
        points_tensor = points_tensor.transpose(1, 2)

        optimizer.zero_grad()

        pred = model(points_tensor)

        loss = criterion(pred, labels_tensor)

        loss.backward()

        optimizer.step()

    scheduler.step(loss)

    #do a validation pass every VAL_CHECK epochs, deterministically (no shuffle)
    if epoch % VAL_CHECK == 0:

        model.eval()
        running_matrix = torch.zeros(23, 23)

        for val_step in range(0, val_shuffle_len, K):

            points, labels = batch_loader(specified_set=validating_batch[val_step:val_step+K])
            points_tensor = torch.tensor(points, dtype=torch.float32)
            labels_tensor = torch.tensor(labels, dtype=torch.int64)
            points_tensor = points_tensor.transpose(1, 2)
            
            with torch.no_grad():

                pred = model(points_tensor)
                pred_classes = torch.argmax(pred, 1)       #take the argmax of the classes dimension to get the predicted class per point
                
                #calculate confusion matrix, add it to a running total for IoU/mIoU calculation
                matrix = calculate_matrix(preds=pred_classes, targets=labels_tensor, num_classes=23, ignore_index=0)
                running_matrix += matrix

        #IoU
        iou, miou = calculate_iou_miou(running_matrix, ignore_index=0)
        print(f"Validation Loop - Epoch {epoch}")

        for j in range(len(iou)):
            print(f"Values for class {j} ==== IOU: {iou[j]:.4f}")
        print(f"MIOU: {miou[j]:.4f}")

        model.train()

        

